# Merchant Overlap: Volume Inflation Hides the Jaccard Signal

This notebook demonstrates the third structural gap — Genie's SQL merchant-overlap query is dominated by high-volume normal accounts, not fraud ring members.

The check asks Genie to find pairs of accounts with the most merchants in common. Fraud ring members share five anchor merchants, giving them high within-ring Jaccard similarity. But the top pairs Genie returns by raw shared-merchant count are dominated by high-volume normal accounts — accounts that visit hundreds of merchants accumulate more absolute overlap than ring members visiting roughly 30 total, even when ring members share a higher fraction of each other's merchant sets.

**Four phases:**

- **Phase 1** — Call Genie and display the result
- **Phase 2** — Load the ground truth ring membership from the UC Volume
- **Phase 3** — Compute the same-ring fraction metric and determine whether the structural gap is confirmed
- **Phase 4** — Explain why raw count inverts the signal and why Jaccard normalization is required

**Before running:** Set your `SPACE_ID` in the configuration cell below. Run `setup/upload_and_create_tables.sh` first to load the tables and upload `ground_truth.json` to the UC Volume.

**Confirmation threshold:** `same_ring_fraction < 0.30` — the gap is confirmed when fewer than 30% of the top-overlap pairs share a fraud ring, showing that raw shared-merchant count is dominated by high-volume normal accounts rather than ring pairs.

In [ ]:
%pip install --upgrade databricks-sdk --quiet
dbutils.library.restartPython()

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Set SPACE_ID to your Genie Space ID before running.
# Find it in Databricks under Genie → your space → the ID in the URL.

SPACE_ID = "YOUR-GENIE-SPACE-ID"   # <-- replace this

CATALOG = "graph-enriched-lakehouse"
SCHEMA  = "graph-enriched-schema"
VOLUME  = "graph-enriched-volume"

In [ ]:
from databricks.sdk import WorkspaceClient
import pandas as pd

from demo_utils import genie_caller, load_ground_truth, build_ring_lookup, check_merchant_overlap

w = WorkspaceClient()
print("Connected to:", w.config.host)

ask_genie = genie_caller(w, SPACE_ID)

## Phase 1 — Call Genie and Display Results

The question below asks Genie to find pairs of accounts with the most merchants in common. This is the merchant-overlap question — the analyst is looking for accounts that visit the same places, which is a signal for coordinated activity.

Genie's natural answer is a self-join on the transactions table: for each pair of accounts, count the distinct merchants both have visited, and return the top 20 pairs by that count. This is the obvious SQL formulation, and it is wrong for the purpose of finding ring members.

The problem is the denominator. Ring members visit roughly 30 merchants total, with 4–5 anchor merchants in common. A normal high-volume account may visit 300 merchants. Two such accounts can accumulate 40–60 shared merchants in absolute terms even though their merchant sets barely overlap proportionally. The raw count measures absolute intersection, not relative overlap. Jaccard similarity — shared merchants divided by the union of both accounts' merchant sets — corrects for visit volume. Raw count does not.

In [ ]:
CHECK_3_QUESTION = (
    "Which pairs of accounts have visited the most merchants in common? "
    "Show me the top 20 pairs by count of shared merchants."
)

print("Question:", CHECK_3_QUESTION)
print()

response = ask_genie(CHECK_3_QUESTION)
print(f"Status:  {response['status']}")

if response["text"] and response["df"] is None:
    print(f"\nGenie returned a text response instead of a table:\n{response['text']}")
    raise RuntimeError(
        "Genie did not return a query result. "
        "Confirm the Genie Space is connected to the correct tables and try again."
    )

print(f"Rows returned: {len(response['df'])}")

In [ ]:
# Genie's answer — the raw result before any comparison
print("Genie returned these account pairs by shared merchant count:\n")
display(response["df"])

In [ ]:
# The SQL Genie generated — shows what Genie actually measured
print("SQL Genie generated:\n")
print(response["sql"])

## Phase 2 — Load Ground Truth Ring Membership

Each fraud ring has five anchor merchants drawn from the full pool of 2,500 merchants. Ring members visit these anchors at an elevated rate, giving them high within-ring Jaccard similarity. `ground_truth.json` records the full ring membership and the anchor merchants for each ring.

Loading this file gives us the ring membership needed to evaluate what fraction of Genie's top-overlap pairs are same-ring accounts.

In [ ]:
GROUND_TRUTH_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/ground_truth.json"

gt = load_ground_truth(GROUND_TRUTH_PATH)
_, whale_ids = build_ring_lookup(gt)

rings             = gt["rings"]
total_rings       = len(rings)
accounts_per_ring = len(rings[0]["account_ids"]) if rings else 0
total_fraud       = sum(len(r["account_ids"]) for r in rings)

print(f"Ground truth loaded from: {GROUND_TRUTH_PATH}")
print(f"  Rings:                {total_rings}")
print(f"  Accounts per ring:    {accounts_per_ring}")
print(f"  Total fraud accounts: {total_fraud:,}")
print(f"  Whale accounts:       {len(whale_ids):,}")
print()
print(f"Confirmation threshold: fewer than 30% of returned pairs share a fraud ring.")
print(f"A confirmed gap shows that raw count is dominated by high-volume normal accounts.")
print()
print("Anchor merchants per ring (first three rings):")
for ring in rings[:3]:
    anchor_ids = [m["merchant_id"] for m in ring["anchor_merchants"]]
    print(f"  Ring {ring['ring_id']}: merchants {anchor_ids}")

## Phase 3 — Compute the Same-Ring Fraction Metric

The metric for this check is `same_ring_fraction`: of the top pairs Genie returned by shared merchant count, what fraction have both accounts in the same fraud ring?

- **Low same-ring fraction (< 0.30):** Fewer than 30% of the top-overlap pairs are same-ring accounts. High-volume normal accounts dominate the ranking. **This confirms the structural gap.** Raw count cannot find ring pairs because volume inflates absolute overlap for normal accounts.
- **High same-ring fraction (≥ 0.30):** Ring pairs appear frequently in the top-20 even by raw count. This would mean the anchor merchants are concentrated enough in absolute terms to dominate the ranking without normalization — the Jaccard gap is narrower than intended. Re-run `setup/verify_fraud_patterns.py` to check the structural properties.

In [ ]:
# Extract pairs from Genie's result.
# Expected columns: account_id_a, account_id_b (and optionally shared_merchant_count)
# If Genie returned different column names, adjust col_a and col_b below.
df = response["df"]

col_a = next(
    (c for c in df.columns if "account_id_a" in c.lower() or (c.lower().endswith("_a") and "account" in c.lower())),
    None,
)
col_b = next(
    (c for c in df.columns if "account_id_b" in c.lower() or (c.lower().endswith("_b") and "account" in c.lower())),
    None,
)

if col_a is None or col_b is None:
    cols = df.columns.tolist()
    col_a, col_b = cols[0], cols[1]
    print(f"Note: could not detect pair columns by name. Using '{col_a}' and '{col_b}'.")
else:
    print(f"Pair columns: '{col_a}' and '{col_b}'")

pairs = list(zip(df[col_a].tolist(), df[col_b].tolist()))
print(f"Pairs extracted: {len(pairs)}")

In [ ]:
ring_lists = [r["account_ids"] for r in gt["rings"]]
result = check_merchant_overlap(pairs, ring_lists)

print("Same-ring fraction analysis:")
print(f"  Total pairs returned:   {result['total_pairs']}")
print(f"  Same-ring pairs:        {result['same_ring_pairs']}")
print(f"  Cross-ring pairs:       {result['cross_ring_pairs']}")
print(f"  Unknown pairs:          {result['unknown_pairs']}")
print(f"  Rings touched:          {result['rings_touched']}")
print()
print(f"  Same-ring fraction:     {result['same_ring_fraction']:.1%}")
print(f"  Confirmation threshold: < 30%")

In [ ]:
print("=" * 62)
print("MERCHANT OVERLAP SUMMARY")
print("=" * 62)
print()
print(f"Genie returned {result['total_pairs']} account pairs ranked by shared merchant count.")
print()
print(f"  Same-ring pairs:    {result['same_ring_pairs']} of {result['total_pairs']}")
print(f"    Ring members share anchor merchants — finding some same-ring pairs")
print(f"    is possible. The question is whether they dominate the top-N.")
print()
print(f"  Same-ring fraction: {result['same_ring_fraction']:.1%}")
print(f"    Confirmation threshold: < 30%")
print()
print("-" * 62)

if result["passed"]:
    print("STRUCTURAL GAP CONFIRMED")
    print("-" * 62)
    print()
    print("Raw shared-merchant count is dominated by high-volume normal accounts.")
    print("Ring pairs, despite sharing 4-5 anchor merchants, do not rank highly")
    print("by absolute count because their total merchant visit volume is low.")
    print("This confirms the gap that Node Similarity closes after GDS enrichment.")
else:
    print("STRUCTURAL GAP NOT CONFIRMED")
    print("-" * 62)
    print()
    print(f"same_ring_fraction {result['same_ring_fraction']:.1%} >= 30%.")
    print("Ring pairs appear too frequently in Genie's top-overlap result.")
    print("The anchor merchants may be concentrated enough to dominate raw counts")
    print("without normalization. Re-run setup/verify_fraud_patterns.py to confirm")
    print("the Jaccard ratio is within target structural properties.")

print()
print("-" * 62)
print("Next: Node Similarity Surfaces Ring Pairs (after GDS enrichment)")
print("-" * 62)
print()
print("After GDS enrichment the accounts table gains a similarity_score column.")
print("Genie can then rank pairs by that Jaccard-derived score — surfacing ring")
print("pairs sharing anchor merchants rather than high-volume normal accounts.")

## Phase 4 — Why Raw Count Inverts the Signal

The gap confirmed by this check is not a limitation of Genie — it is a property of the metric.

### The volume inflation problem

Raw shared-merchant count measures absolute intersection: how many distinct merchants appear in both accounts' transaction histories. This measure is correct if every account visits roughly the same number of merchants. It breaks when visit volume varies.

In this dataset:

- **Fraud ring members** visit roughly 30 distinct merchants. Five of those are shared anchor merchants for their ring. The absolute count of shared merchants with another ring member is approximately 4–5.
- **High-volume normal accounts** may visit 200–400 distinct merchants. Two such accounts, each visiting a broad cross-section of the merchant pool, accumulate 40–80 shared merchants in absolute terms — not because they are coordinating, but because large merchant sets overlap by chance at scale.

The result: Genie's raw-count ranking surfaces normal accounts at the top, and ring pairs — which have much higher proportional overlap — are buried below them.

### What Jaccard corrects

Jaccard similarity is defined as:

```
Jaccard(A, B) = |A ∩ B| / |A ∪ B|
```

For a ring pair sharing 5 of 30 total merchants: `5 / (30 + 30 - 5) = 5 / 55 ≈ 0.09`.

For two high-volume normals sharing 60 of 400 total merchants: `60 / (400 + 400 - 60) = 60 / 740 ≈ 0.08`.

The Jaccard scores are similar in that simplified example — but the five anchor merchants that ring members share are specific: the same five merchants, consistently, across the full 100-member ring. Node Similarity exploits this within a bipartite account-merchant projection: it scores every account pair by Jaccard over their merchant neighborhoods simultaneously. Ring pairs score high because they share the same dense anchor-merchant neighborhood. Normal accounts score lower because their shared merchants are distributed across a much larger union.

### Why a skilled analyst could write this in SQL — and what GDS adds

Unlike PageRank and Louvain, Jaccard similarity is computable in SQL. A skilled analyst who knows to normalize by union size could write the correct query. The GDS advantage here is different:

- **Scale:** Node Similarity scores every account pair in the bipartite graph simultaneously without the analyst knowing which pairs to examine.
- **Discovery:** The analyst does not need to hypothesize that merchant-set overlap is the signal. GDS computes it globally and writes the result as a column that Genie can query directly.
- **The demo contrast:** Check 3 shows the intuitive SQL formulation — the one Genie writes — producing a result dominated by the wrong accounts. Check 6 shows the GDS column producing ring pairs at the top. The before/after is the gap between the obvious metric and the correct one.